In [2]:
import aerosandbox as asb 
import casadi as ca 



In [10]:
cst_upper=ca.MX.sym("cst_upper",8)
cst_lower=ca.MX.sym("cst_lower",8)
cst_le=ca.MX.sym("le")

init_af=asb.KulfanAirfoil("naca0012")

new_af=asb.KulfanAirfoil(
    upper_weights=init_af.upper_weights*(1+cst_upper),
    lower_weights=init_af.lower_weights*(1+cst_lower),
    leading_edge_weight=init_af.leading_edge_weight*(1+cst_le)
)

aero=new_af.get_aero_from_neuralfoil(alpha=2,Re=1e6,mach=0.3)
CL=aero["CL"]
CD=aero["CD"]

aero_solver=ca.Function("aero",[cst_upper,cst_lower,cst_le],[CL,CD],["cst_upper","cst_lower","cst_le"],["CL","CD"])
aero_solver_jac=aero_solver.jacobian()
aero_solver_hess=aero_solver_jac.jacobian()

# codegen=ca.CodeGenerator("aero.c")
# codegen.add(aero_solver)
# codegen.add(aero_solver_jac)
# codegen.add(aero_solver_hess)
# codegen.generate()

ca.GraphBuilder(aero_solver).export_onnx("aero.onnx")

In [18]:
onnx_aero=ca.GraphBuilder("aero.onnx").create("aero",{"symbolic":True})
onnx_aero_jac=onnx_aero.jacobian()
onnx_aero_hess=onnx_aero_jac.jacobian()

codegen=ca.CodeGenerator("aero_onnx.c")
codegen.add(onnx_aero)
codegen.add(onnx_aero_jac)
codegen.add(onnx_aero_hess)
codegen.generate()

'aero_onnx.c'

In [ ]:
onnx_aero(cst_upper=ca.DM.zeros(8),cst_lower=ca.DM.zeros(8),cst_le=ca.DM.zeros(1))


Function(jac_aero:(cst_upper[8],cst_lower[8],cst_le,out_CL[1x1,0nz],out_CD[1x1,0nz])->(jac_CL_cst_upper[1x8],jac_CL_cst_lower[1x8],jac_CL_cst_le,jac_CD_cst_upper[1x8],jac_CD_cst_lower[1x8],jac_CD_cst_le) MXFunction)

In [4]:
import casadi as cs
import torch
import l4casadi as l4c


class MultiLayerPerceptron(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.input_layer = torch.nn.Linear(2, 512)

        hidden_layers = []
        for i in range(20):
            hidden_layers.append(torch.nn.Linear(512, 512))

        self.hidden_layer = torch.nn.ModuleList(hidden_layers)
        self.out_layer = torch.nn.Linear(512, 1)

    def forward(self, x):
        x = self.input_layer(x)
        for layer in self.hidden_layer:
            x = torch.tanh(layer(x))
        x = self.out_layer(x)
        return x


pyTorch_model = MultiLayerPerceptron()

In [5]:
l4c_model = l4c.L4CasADi(pyTorch_model,name="zcc",generate_jac_jac=True, device='cpu')  # device='cuda' for GPU
x_sym = cs.MX.sym('x', 1, 2)

In [6]:
y_sym = l4c_model(x_sym) 

d:\micromamba\envs\test\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
d:\micromamba\envs\test\Lib\ast.py:407: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  return visitor(node)
d:\micromamba\envs\test\Lib\site-packages\torch\jit\_freeze.py:106: FutureWarning: `torch.jit.freeze` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
d:\micromamba\envs\test\Lib\site-packages\torch\jit\_freeze.py:230: FutureWarning: `torch.jit.optimize_for_inference` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


: 

In [12]:
import casadi as ca 
x=ca.MX.sym("x",2)
y=ca.MX.sym("y",3)
z=ca.sum(x)+ca.sum(y)
func_zcc=ca.Function("zcc",[x,y],[z],["x","y"],["z"])
func_zcc_jac=func_zcc.jacobian()
func_zcc_jac_jac=func_zcc_jac.jacobian()

cg=ca.CodeGenerator("zcc.c",{"cpp":False})
cg.add(func_zcc)
cg.add(func_zcc_jac)
cg.add(func_zcc_jac_jac)
cg.generate()

'zcc.c'

In [3]:
import casadi as ca 
xxx=ca.external("zcc","./_l4c_generated/zcc.dll")

RuntimeError: .../casadi/core/casadi_os.cpp:335: Assertion "handle!=nullptr" failed:
DllLibrary::init_handle: Cannot load shared library './_l4c_generated/zcc.dll': 
   (
    Searched directories: 1. CASADI_PLUGIN_SEARCH_PATH env var
                          2. casadipath from GlobalOptions
                          3. CASADIPATH env var
                          4. PATH env var (Windows)
                          5. LD_LIBRARY_PATH env var (Linux)
                          6. DYLD_LIBRARY_PATH env var (osx)
    A library may be 'not found' even if the file exists:
          * library is not ABI-compatible (different compiler/bitness)
          * the dependencies are not found
          * the dependencies are found but have an ABI-incompatible version/compiler/bitness
   )
  Tried 'd:\micromamba\envs\test\Lib\site-packages\casadi' :
    Error code (WIN32): 126
  Tried '' :
    Error code (WIN32): 126
  Tried '.' :
    Error code (WIN32): 126
  Tried '.\_l4c_generated' :
    Error code

In [11]:
import os 
os.environ["TESTFLO_RUNNING"]="false"
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid


prob = om.Problem(reports=None)
model = prob.model

model.add_subsystem('comp', Paraboloid(), promotes=['*'])

prob.driver = om.modOptDriver()
prob.driver.options['optimizer'] = 'SLSQP'
prob.driver.options['maxiter'] = 200
prob.driver.options['disp'] = False
prob.driver.options['turn_off_outputs']=True

model.add_design_var('x', lower=-50.0, upper=50.0)
model.add_design_var('y', lower=-50.0, upper=50.0)
model.add_objective('f_xy')

prob.setup()

prob.set_val('x', 50.0)
prob.set_val('y', 50.0)

prob.run_driver()

Setting objective name as "f_xy".


Problem: problem11
Driver:  modOptDriver
  success     : True
  iterations  : 6
  runtime     : 1.8884E-02 s
  model_evals : 6
  model_time  : 1.3315E-03 s
  deriv_evals : 0
  deriv_time  : 0.0000E+00 s
  exit_status : SUCCESS